# Ejercicio 4. Agente de mantenimiento

**Enunciado:** Desarrollar un agente que busque repuestos por código, consulte almacén, genere una orden por falla y equipo y actualice el historial de fallas.

Fuente: Martínez P. (2026), sección 7.11.

## Cómo utilizar este notebook

Abre el archivo en Jupyter o VS Code con un kernel Python del mismo entorno del proyecto. Colócalo en la raíz o en `notebooks/`. Las celdas de texto contienen la respuesta al ejercicio; las de código generan evidencia al ejecutarlas en tu computador.

No se han ejecutado las consultas contra tu Ollama ni tu base de datos. Los resultados esperados no se presentan como resultados obtenidos. Si hace falta instalar Jupyter, utiliza en tu entorno `python -m pip install notebook ipykernel`; las dependencias de la aplicación se instalan desde `requirements.txt`.

Las rutas y funciones corresponden al código desarrollado en la conversación. Los manuales concretos, los modelos instalados y el contenido real de tus skills deben verificarse localmente.

## Respuesta

El agente utiliza llamadas a herramientas: el modelo solicita una función con argumentos y Python ejecuta únicamente funciones permitidas. Después, el resultado se incorpora a la conversación para que el modelo formule su respuesta. Ollama documenta este intercambio mediante `tools`, `tool_calls` y mensajes con rol `tool` (Ollama, s. f.-b).

### Correspondencia con el enunciado

| Función solicitada | Implementación | Datos utilizados |
|---|---|---|
| Buscar un repuesto por código | `buscar_repuesto()` en `tools/spare_parts.py` | `repuestos`, `equipos_repuestos` y `equipos` |
| Consultar almacén por código | `consultar_inventario()` en `tools/inventory.py` | Catálogo e inventario |
| Generar una orden por falla y equipo | `crear_orden_trabajo()` en `tools/work_orders.py` | Equipo registrado y descripción de falla |
| Actualizar historial de fallas | Inserción conjunta en `crear_orden_trabajo()` | Nueva falla vinculada a la orden |
| Consultar el historial | `consultar_historial()` en `tools/failure_history.py` | Fallas y órdenes asociadas |

La coordinación se encuentra en `agents/maintenance_agent.py`. `llm/ollama_client.py` proporciona `chat_message()` para conservar solicitudes de herramientas y `chat()` para las consultas de texto del RAG y del modo general.

SQLite se prepara con `database/schema.sql` y `database/database.py`. Los datos ficticios se cargan con `database/seed_demo.py`; sus códigos tienen el prefijo `DEMO-`. Las compatibilidades de demostración no son afirmaciones técnicas sobre piezas reales.

### Consistencia y reintentos

La orden, la falla y su identificador de solicitud se guardan en una misma transacción. La tabla `solicitudes_orden`, añadida mediante `database/migrate_idempotency.py`, relaciona un identificador con una orden.

Un reintento con el mismo identificador y los mismos datos recupera la orden existente. El mismo identificador con datos distintos produce un conflicto. Un identificador nuevo representa una solicitud nueva, aunque el texto de la falla sea idéntico.

### Demostración

```bash
python -m database.seed_demo
python -m database.migrate_idempotency
python -m agents.maintenance_agent "Busca DEMO-BRG-6205 y consulta sus existencias."
```

Para crear una orden se habilita explícitamente la herramienta de escritura:

```bash
python -m agents.maintenance_agent "Crea una orden para DEMO-P-101 por vibración elevada, con prioridad ALTA." --permitir-orden --solicitud-id demo-taller-001
python -m tools.failure_history DEMO-P-101
```

Repetir el comando de creación con el mismo identificador debe devolver la misma orden sin duplicar la falla. Esto es un resultado esperado que debe comprobarse en la ejecución.

En `app.py`, `/agente` permite consultas; la creación de órdenes se demuestra con el comando de terminal anterior. El agente procesa una solicitud a la vez y limita las rondas y llamadas. La memoria conversacional existe durante cada ejecución del agente; el historial de fallas se conserva en SQLite.

El agente no modifica equipos físicos, no reserva ni descuenta repuestos y no consulta el RAG como herramienta. La consulta documental y el agente comparten el proyecto, pero se ejecutan como modos separados.

## Demostración con datos ficticios

Las celdas de preparación y escritura modifican la base configurada en tu proyecto. Usa la base de demostración, no datos operativos.

In [ ]:
from pathlib import Path
import sys

# Coloca los notebooks en la raíz del proyecto o en su carpeta notebooks/.
# Si están en otro lugar, escribe aquí la ruta de tu proyecto.
PROJECT_DIR = None  # Ejemplo: Path(r"C:\industrial-ai-assistant")

if PROJECT_DIR is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next(
        (p for p in candidates if (p / "config.py").is_file()
         and (p / "app.py").is_file()), None
    )
if PROJECT_DIR is None:
    raise FileNotFoundError("Indica PROJECT_DIR con la ruta de tu proyecto.")
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
if not (PROJECT_DIR / "config.py").is_file():
    raise FileNotFoundError("La ruta indicada no contiene config.py.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Proyecto:", PROJECT_DIR)

In [ ]:
RUN_DEMO_SETUP = False
if RUN_DEMO_SETUP:
    from database.seed_demo import seed_demo
    from database.migrate_idempotency import migrate
    migrate()
    seed_demo()
    print("Datos de demostración preparados.")
else:
    print("Preparación sin ejecutar.")

In [ ]:
RUN_READS = False
if RUN_READS:
    from tools.spare_parts import buscar_repuesto
    from tools.inventory import consultar_inventario
    from tools.failure_history import consultar_historial
    print(buscar_repuesto("DEMO-BRG-6205"))
    print(consultar_inventario("DEMO-BRG-6205"))
    print(consultar_inventario("DEMO-SEAL-001"))
    print(consultar_inventario("DEMO-NO-EXISTE"))
    print(consultar_historial("DEMO-P-101"))
else:
    print("Consultas directas sin ejecutar.")

In [ ]:
RUN_AGENT_QUERY = False
if RUN_AGENT_QUERY:
    from agents.maintenance_agent import run_agent
    agent_result = run_agent(
        "Busca el repuesto DEMO-BRG-6205 y consulta sus existencias."
    )
    print(agent_result["answer"])
    print("Ejecuciones reales:", agent_result["executions"])
else:
    print("Agente de consultas sin ejecutar.")

### Crear una orden mediante el agente

Esta celda sí puede insertar una orden y su falla. Conserva `REQUEST_ID` al reintentar. Una solicitud diferente requiere otro identificador. Si el LLM falla después de la escritura, revisa el historial antes de reintentar. Verifica que has aplicado la versión final de `maintenance_agent.py` y la migración.

In [ ]:
RUN_WRITE = False
REQUEST_ID = "notebook-ejercicio-4-001"
write_result = None

if RUN_WRITE:
    from agents.maintenance_agent import run_agent
    print("Identificador que debes conservar:", REQUEST_ID)
    write_result = run_agent(
        "Crea una orden para DEMO-P-101 por vibración elevada, con prioridad ALTA.",
        allow_create=True,
        request_id=REQUEST_ID,
    )
    print(write_result["answer"])
    print(write_result["executions"])
else:
    print("Escritura sin ejecutar.")

### Comprobar el reintento y el historial

Se reutilizan los argumentos que realmente ejecutó el agente, para no introducir diferencias en la descripción de la falla. La comprobación de reintento se hace directamente sobre la herramienta; no es una segunda ejecución del agente.

In [ ]:
if RUN_WRITE and write_result is not None:
    from tools.work_orders import crear_orden_trabajo
    from tools.failure_history import consultar_historial
    writes = [e for e in write_result["executions"]
              if e["tool"] == "crear_orden_trabajo"
              and e["result"].get("ok")
              and e["result"].get("resultado", {}).get("procesada")]
    if not writes:
        raise RuntimeError("No hay evidencia de una orden procesada por el agente.")
    execution = writes[0]
    original = execution["result"]["resultado"]
    retry = crear_orden_trabajo(**execution["arguments"], solicitud_id=REQUEST_ID)
    assert retry["reutilizada"] is True
    assert retry["orden_id"] == original["orden_id"]
    assert retry["historial_id"] == original["historial_id"]
    print("Reintento verificado:", retry)
    print("Historial:", consultar_historial(original["equipo_codigo"]))
else:
    print("Comprobación de escritura sin ejecutar.")

## Interpretación

La evidencia principal está en los resultados de herramientas y los registros SQLite. El modelo no debe presentar una creación como realizada si no obtuvo `procesada=true`. Un historial vacío significa ausencia de registros, no ausencia histórica de fallas.

El enunciado de actualizar el historial se cumple mediante la inserción conjunta de la falla y la orden. No se necesita una segunda herramienta que vuelva a insertar la misma falla.

## Referencias

AI Hero. (s. f.). *Become a real AI Hero*. https://www.aihero.dev/

Chroma. (s. f.-a). *Configure collections*. https://docs.trychroma.com/docs/collections/configure

Chroma. (s. f.-b). *Query and get*. https://docs.trychroma.com/docs/querying-collections/query-and-get

Espressif Systems. (s. f.). *Support for external RAM: ESP32-S3*. ESP-IDF Programming Guide. https://docs.espressif.com/projects/esp-idf/en/stable/esp32s3/api-guides/external-ram.html

Martínez P., J. J. (2026, septiembre). *7. Inteligencia artificial generativa* [Material de curso proporcionado en el archivo `7.md`].

Ollama. (s. f.-a). *Generate embeddings*. https://docs.ollama.com/api/embed

Ollama. (s. f.-b). *Tool calling*. https://docs.ollama.com/capabilities/tool-calling

pypdf. (s. f.). *Extract text from a PDF*. https://pypdf.readthedocs.io/en/stable/user/extract-text.html